# Environment & Git Setup Notes

Personal reference for how this project's conda environment and GitHub repo were set up, plus the activation fix needed for PowerShell. Commands are PowerShell / Git commands, not Python — copy-paste into a terminal, don't run these cells.

## 1. Create the environment in VS Code

Despite the folder being named `venv`, this was created as a **conda environment** (via `-p`, i.e. `--prefix`), not a standard Python `venv`. That distinction matters later for activation.

```powershell
conda create -p venv python==3.8 -y
conda activate venv/
```

## 2. Initialize git and make the first commit

```powershell
git init
git add README.md
git commit -m "first commit"
git status
```

## 3. Rename branch and connect to GitHub

```powershell
git branch -M main
git remote add origin https://github.com/Satya666-pra/MLPROJECT.git
git remote -v
git config --global user.email
git push -u origin main
```

## 4. Sync remote changes

```powershell
git pull
```

## 5. Gotcha: `pip install` fails with "externally-managed-environment"

This happened when `pip install -r requirements.txt` was run without the environment properly activated in that terminal — pip fell back to the protected base/system Python instead of the project's env.

**Root cause found:** `conda activate` alone did nothing in a plain PowerShell session because conda's shell hook wasn't initialized there yet (`conda` resolved to the raw `.exe`, not the shell-integrated function). So `$env:CONDA_PREFIX` never got set, and `python`/`pip` kept pointing outside the env.

**One-time permanent fix:**
```powershell
conda init powershell
```
Then **close and reopen the terminal**. After that, plain `conda activate` works in every new PowerShell window.

**If you ever see the hook missing again** (e.g. a terminal opened in a weird way), load it manually for that one session:
```powershell
(& "C:\Users\satya\miniconda3-pbe\Scripts\conda.exe" "shell.powershell" "hook") | Out-String | Invoke-Expression
conda activate ".\venv"
```

## 6. Correct day-to-day workflow (after the one-time fix above)

```powershell
conda activate ".\venv"
pip install -r requirements.txt
```

**How to verify the env is actually active before installing anything:**
```powershell
$env:CONDA_PREFIX        # should show the path ending in \venv
python --version         # should show Python 3.8.0
pip --version             # path shown should be inside \venv\lib\site-packages\pip
```

## 7. What `setup.py` does and why it exists

`setup.py` is what turns this folder into an installable Python **package** (named `mlproject`) instead of just a loose collection of scripts. It's what runs behind the scenes when `pip install -e .` is executed.

```python
from setuptools import find_packages, setup
from typing import List

HYPEN_E_DOT = '-e .'
def get_requirements(file_path: str) -> List[str]:
    '''
    this function will return the list of requirements
    '''
    requirements = []
    with open(file_path) as file_obj:
        requirements = file_obj.readlines()
        requirements = [req.replace("\n", "") for req in requirements]

        if HYPEN_E_DOT in requirements:
            requirements.remove(HYPEN_E_DOT)

    return requirements

setup(
    name='mlproject',
    version='0.0.1',
    author='SatyaprakashKabi',
    author_email='satyaprakashKabi@gmail.com',
    packages=find_packages(),
    install_requires=get_requirements('requirements.txt')
)
```

**Line-by-line:**

- `from setuptools import find_packages, setup` — `setup()` is the function that actually registers the package's metadata with pip/setuptools. `find_packages()` auto-discovers every directory containing an `__init__.py` (i.e. every importable sub-package) so they don't have to be listed by hand.

- `HYPEN_E_DOT = '-e .'` — a constant holding the literal text `-e .`, the line that appears at the bottom of `requirements.txt`. `-e .` tells pip "install *this* project itself, in editable mode" (see below). It's a pip-only instruction, not a real package name, so it must never be passed to `install_requires`.

- `get_requirements(file_path)` — reads `requirements.txt` and turns it into the list of dependency strings `setup()` needs:
  - `file_obj.readlines()` reads every line, including its trailing `\n`.
  - the list comprehension strips those `\n` characters off each line.
  - the `if HYPEN_E_DOT in requirements` check removes the `-e .` entry if present — **this is the important guard**. Without it, `setup()` would try to treat `-e .` as a package name and installation would break, because `-e .` only makes sense as a pip command-line flag, not as an item in `install_requires`.

- `setup(...)` — the actual package registration:
  - `name='mlproject'` — the package's importable/installed name.
  - `version='0.0.1'` — package version.
  - `author`, `author_email` — metadata only, no functional effect.
  - `packages=find_packages()` — the list of sub-packages to include (auto-detected).
  - `install_requires=get_requirements('requirements.txt')` — the dependency list, reusing `requirements.txt` as the single source of truth instead of duplicating package names in two places.

**How this connects back to `requirements.txt` and the earlier `pip install` step:**

`requirements.txt` ends with `-e .`. That line is what made `pip install -r requirements.txt` (run in Section 6) also install *this project itself* as an editable package — which is why the install log showed `Running setup.py develop for mlproject` and `Successfully installed ... mlproject`. "Editable" means the package is linked back to this source folder rather than copied into `site-packages`, so any code changes here are picked up immediately without reinstalling — useful during active development, and why `from src...` / `import mlproject...`-style imports work project-wide once the env is activated.

## 8. Gotcha: `src/__init__.py` missing from `SOURCES.txt`

`mlproject.egg-info/SOURCES.txt` is auto-generated every time `pip install -e .` runs, listing whatever `find_packages()` discovers at that moment. At one point it showed `src/__init__.py` on line 8, but the file on disk had disappeared — it had been created locally at some earlier point, never committed to git, and then got deleted, so the next reinstall regenerated `SOURCES.txt` without it.

**Fix — recreate the package folder and reinstall:**
```powershell
mkdir src
New-Item src\__init__.py -ItemType File

conda activate ".\venv"
pip install -e . --no-deps

Get-Content mlproject.egg-info\SOURCES.txt   # confirm src/__init__.py is listed again
```
`--no-deps` skips reinstalling pandas/numpy/etc. — it just re-registers the local package so `find_packages()` picks up the new `src/` folder.

## 9. Gotcha: `.gitignore` rules weren't being applied

`venv/` and `mlproject.egg-info/` kept showing up as untracked in `git status`, even though the ignore file already listed `venv/` and `*.egg-info/`. The cause: the file was named **`gitignore`** (no leading dot) instead of **`.gitignore`** — Git only recognizes the dotted filename, so the rules were silently never applied.

**Fix — rename it and re-check status:**
```powershell
git mv gitignore .gitignore
git status
```
If `git mv` errors with `fatal: bad source` (can happen on Windows), do it as a plain rename plus a manual stage instead — git still detects it as a rename once both paths are staged together:
```powershell
Rename-Item gitignore .gitignore
git add gitignore .gitignore
git status   # should show "renamed: gitignore -> .gitignore"
```

## 10. Committing and pushing only the real source files

Never commit `venv/` (the local environment) or `mlproject.egg-info/` (generated build metadata) — both are covered by `.gitignore` once Section 9's fix is applied. Stage only the actual project files:

```powershell
git add .gitignore README.md environment_setup_notes.ipynb requirements.txt setup.py src/
git status                          # sanity check — venv/ and egg-info/ should NOT appear
git commit -m "Add src package, setup.py, requirements, and rename gitignore to .gitignore"
git push
```

## 11. Gotcha: `python src\exception.py` works from VS Code's Run button but not from the terminal

- **Run ▶ button works** because VS Code invokes the *selected interpreter's* `python.exe` (`venv\python.exe`) directly, bypassing the shell entirely. That interpreter already has `mlproject` installed (editable install from Section 7), so `import src...` succeeds.
- **Terminal fails** with `ModuleNotFoundError: No module named 'src'` because a fresh PowerShell terminal auto-activates conda's **`base`** environment (default `conda init` behavior), not this project's `venv`. Base has a different Python with no `mlproject` installed.

**Fix — always activate the project env explicitly before running scripts in a terminal:**
```powershell
conda activate .\venv
python src\exception.py
```
Don't rely on VS Code's auto-activation settings (`python.terminal.activateEnvironment`, the newer `python-envs.*` settings) to always pick the right env automatically — they can conflict with each other. Explicitly running `conda activate .\venv` in the terminal is the reliable habit.

**Also fixed:** pinned `.vscode/settings.json` so the workspace defaults to `venv\python.exe`:
```json
{
    "python.defaultInterpreterPath": "${workspaceFolder}\\venv\\python.exe",
    "python.terminal.activateEnvironment": true
}
```
If this default doesn't seem to apply, manually run **Ctrl+Shift+P → "Python: Select Interpreter"** and pick `venv\python.exe` — VS Code only auto-applies the default when no interpreter has been manually chosen before for this workspace.

## 12. Deep dive: `src/logger.py`

```python
import logging
import os
from datetime import datetime

LOG_FILE = f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"
logs_path = os.path.join(os.getcwd(), "logs")
os.makedirs(logs_path, exist_ok=True)

LOG_FILE_PATH = os.path.join(logs_path, LOG_FILE)

logging.basicConfig(
    filename=LOG_FILE_PATH,
    format="[ %(asctime)s ] %(lineno)d %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)
```

**Purpose:** a shared, pre-configured logger that every other module in `src/` can import, so log formatting and destination are defined in exactly one place.

**Line-by-line:**

- `LOG_FILE = f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"` — builds a filename from the current timestamp (e.g. `09_26_2026_17_01_51.log`), so **every run of the program gets its own fresh log file** rather than all runs appending to one shared file.

- `logs_path = os.path.join(os.getcwd(), "logs")` — the target directory is `logs/` under the **current working directory** (`os.getcwd()`), not necessarily the project root. This matters: if you run a script from a different folder, `os.getcwd()` changes, and a *new* `logs/` folder appears wherever you happened to run the command from. (This is why you may have noticed log folders showing up in unexpected places if scripts were run from different directories.)

- `os.makedirs(logs_path, exist_ok=True)` — creates the `logs/` folder if it doesn't already exist. `exist_ok=True` prevents an error on subsequent runs when the folder is already there.

- `LOG_FILE_PATH = os.path.join(logs_path, LOG_FILE)` — the full path to this run's log file, e.g. `.../logs/09_26_2026_17_01_51.log`.
  - **Bug that was fixed here (see Section 13-equivalent earlier troubleshooting):** the original code passed `LOG_FILE` (already ending in `.log`) into `os.path.join(os.getcwd(), "logs", LOG_FILE)` and then called `os.makedirs()` on *that full file path* — which created a **directory** named like a log file (e.g. `logs/09_26_2026_16_58_55.log/`), and then wrote the actual log file *inside* that oddly-named folder. The fix was to make `logs_path` just the `logs` directory, and only join in `LOG_FILE` afterward for the actual file path.

- `logging.basicConfig(...)` — configures Python's built-in **root logger** (the shared, global logger object) exactly once for the whole process:
  - `filename=LOG_FILE_PATH` — sends log output to this file instead of the console.
  - `format="[ %(asctime)s ] %(lineno)d %(name)s - %(levelname)s - %(message)s"` — defines what each log line looks like: timestamp, the line number the log call was made from, the logger's name (`root` by default here), the severity level (INFO/WARNING/ERROR/...), and the actual message.
  - `level=logging.INFO` — the minimum severity that gets recorded. `DEBUG` messages would be *silently ignored*; `INFO`, `WARNING`, `ERROR`, and `CRITICAL` all get written.

**Important gotcha with `basicConfig`:** it only has an effect the **first time** it's called in a given Python process. Since this configuration runs automatically the moment `logger.py` is imported (there's no `if __name__ == "__main__"` guard around it), the very first `from src.logger import logging` anywhere in the program sets up the file/format/level for the *entire run* — every other module that later does `from src.logger import logging` just reuses that same already-configured root logger.

## 13. Deep dive: `src/exception.py`

```python
import sys
from src.logger import logging

def error_message_detail(error, error_detail: sys):
    _, _, exc_tb = error_detail.exc_info()
    file_name = exc_tb.tb_frame.f_code.co_filename
    error_message = "Error occured in python script name [{0}] line number [{1}] error message[{2}]".format(
        file_name, exc_tb.tb_lineno, str(error))
    return error_message

class CustomException(Exception):
    def __init__(self, error_message, error_detail: sys):
        super().__init__(error_message)
        self.error_message = error_message_detail(error_message, error_detail=error_detail)

    def __str__(self):
        return self.error_message

if __name__ == "__main__":
    try:
        a = 1 / 0
    except Exception as e:
        logging.info(error_message_detail(e, sys))
        raise CustomException(e, sys)
```

**Purpose:** a single, project-wide custom exception class. The idea (standard in this ML project template) is that *every* `except` block anywhere in the codebase re-raises the caught error as `CustomException(e, sys)`, so no matter where something breaks, the final error message always tells you the exact file, the exact line, and the original error text — instead of a bare, generic traceback.

**Line-by-line:**

- `import sys` — needed for `sys.exc_info()`, Python's way of asking "what exception is currently being handled?"

- `from src.logger import logging` — importing this line is what actually **triggers** all of `logger.py`'s module-level code (Section 12) the first time it runs, wiring up the log file destination/format for the whole program.

- `error_message_detail(error, error_detail: sys)` — builds the descriptive message string:
  - `error_detail.exc_info()` — here `error_detail` is the `sys` module itself (passed in by the caller as `sys`), and `sys.exc_info()` returns a 3-tuple `(exception_type, exception_value, traceback_object)` describing whatever exception is currently being handled.
  - `_, _, exc_tb = ...` — the type and value are discarded (`_`); only the **traceback object** (`exc_tb`) is kept, because that's what holds the file/line info.
  - `exc_tb.tb_frame.f_code.co_filename` — drills into the traceback: `tb_frame` is the stack frame where the exception occurred, `f_code` is that frame's compiled code object, and `co_filename` is the source filename it belongs to.
  - `exc_tb.tb_lineno` — the exact line number in that file where the exception was raised.
  - the `.format(...)` call assembles all three pieces — filename, line number, and `str(error)` (the original exception's message) — into one human-readable string.

- `class CustomException(Exception)` — a custom exception type that wraps the original one with extra context:
  - `super().__init__(error_message)` — calls the base `Exception.__init__`, storing whatever was passed as `error_message` (note: in this codebase's usage, that's actually the *original exception object* `e`, not a string — slightly confusing naming, but harmless since it's immediately overwritten below).
  - `self.error_message = error_message_detail(error_message, error_detail=error_detail)` — immediately replaces that with the fully detailed string built above, and stores it as an instance attribute.
  - `def __str__(self): return self.error_message` — overrides how the exception renders as text. This is why, when `CustomException` is left uncaught, Python's default traceback printer (which calls `str()` on the exception) shows the **detailed** message rather than Python's default repr of the wrapped exception.

- `if __name__ == "__main__": ...` — a self-test block that **only runs when this file is executed directly** (`python src\exception.py`), not when it's imported by another module elsewhere in the project:
  - `a = 1 / 0` — deliberately triggers a `ZeroDivisionError` to have something to catch.
  - `logging.info(error_message_detail(e, sys))` — writes the same detailed message into the log file (Section 12's logger). This was a deliberate fix — the original code only logged the generic string `"Divide by zero error"`, which wasn't useful for debugging later since it lacked file/line/error detail.
  - `raise CustomException(e, sys)` — wraps the caught `ZeroDivisionError` and re-raises it. Since nothing catches it further up, Python prints the full chained traceback (`ZeroDivisionError` → "During handling of the above exception, another exception occurred" → `CustomException: Error occured in python script name [...] line number [...] error message[...]`).